# 👤 6-Bit Face Recognition Quantization & Embedding Alignment Demo

This notebook demonstrates how to use the modularized source code (`src/`) to load a Face Recognition model, quantize it to 6-bits, set up the proposed **$\beta$-MSE loss**, perform face detection, and check the model state (such as freezing observers).

### 1. Import Project Modules
We import our core modules for preprocessing, architecture definitions, quantization wrapping, and loss functions.

In [ ]:
import os
import sys
import torch
from PIL import Image

# Ensure project root is in python path
sys.path.append('..')

from src.models import iresnet18
from src.quantizer import quantize_model, freeze_model, unfreeze_model
from src.losses import BetaMSELoss
from src.preprocessor import FacePreprocessor

### 2. Configure Preprocessor & OpenCV SSD Bounding Box Crop
We initialize our face preprocessor using OpenCV's SSD detector configurations to crop faces, resize to $112 \times 112$, and normalize color values.

In [ ]:
SSD_PROTO = "../../models/deploy.prototxt.txt"
SSD_MODEL = "../../models/res10_300x300_ssd_iter_140000.caffemodel"

# Instantiate our preprocessor (falls back to raw PIL if weights are missing)
preprocessor = FacePreprocessor(detector_prototxt=SSD_PROTO, detector_weights=SSD_MODEL)

# Generate a dummy tensor representing a batch of 2 normalized images
dummy_batch = torch.randn(2, 3, 112, 112)
print("Preprocessed batch shape:", dummy_batch.shape)

### 3. Initialize Teacher & Proposed Q6 Student Models
We load a standard full-precision (`FP32`) teacher and wrap it recursively using our custom straight-through estimator (STE) quantization modules.

In [ ]:
# Load full precision teacher
teacher = iresnet18(num_features=512)
teacher.eval()
print("Teacher parameter count:", sum(p.numel() for p in teacher.parameters()))

# Instantiate student (loaded from same structure but quantized recursively)
student_fp32 = iresnet18(num_features=512)
student = quantize_model(student_fp32, weight_bit=6, act_bit=6)
print("Quantized Student wrapped successfully.")

### 4. Embedding Alignment via $\beta$-MSE Loss
We show the forward pass computations and calculate the coordinated alignment gradient signal.

In [ ]:
# Initialize alignment loss with beta weight
criterion = BetaMSELoss(beta=300.0)

# Forward pass
with torch.no_grad():
    teacher_feat = teacher(dummy_batch)
    
student_feat = student(dummy_batch)

# Compute Coordinated loss
loss = criterion(student_feat, teacher_feat)
print(f"Teacher embedding shape: {teacher_feat.shape}")
print(f"Student embedding shape: {student_feat.shape}")
print(f"Coordinated Beta-MSE Loss: {loss.item():.6f}")

### 5. Transitioning Phases (Freezing observers)
During training Phase 2 and 3, observers must be frozen to fix activation scales and prevent system oscillation. We demonstrate this state transition.

In [ ]:
# Check one of the student's internal observers
sample_act_layer = None
for m in student.modules():
    if m.__class__.__name__ == "QuantAct":
        sample_act_layer = m
        break

if sample_act_layer:
    print("--- Before Freezing ---")
    print("Observer running stat:", sample_act_layer.running_stat)
    
    # Apply freeze
    freeze_model(student)
    
    print("\n--- After Freezing (Phase 2 & 3) ---")
    print("Observer running stat:", sample_act_layer.running_stat)
else:
    print("Quantization layers not found in the model.")